Imports

In [5]:
import duckdb
import pandas as pd
import os
print("Libraries loaded")

/Users/manoharnaddunoori/BD&BI_Capstone/notebooks
['.DS_Store', 'IMDB TMDB movie dataset.csv', '.ipynb_checkpoints', 'movies_cleaned.csv']
Libraries loaded


Configuration

In [8]:
RAW_FILE     = '../data/IMDB TMDB Movie Dataset.csv'
CLEAN_CSV    = '../data/movies_cleaned.csv'
CLEAN_PARQUET = '../data/movies_cleaned.parquet'

print("Config set")

Config set


Extract

In [9]:
print("Step 1: Extracting raw data...")

con = duckdb.connect()

con.execute(f"""
    CREATE TABLE movies_raw AS
    SELECT * FROM read_csv_auto(
        '{RAW_FILE}',
        ignore_errors=True
    )
""")

raw_count = con.execute("SELECT COUNT(*) FROM movies_raw").fetchone()[0]
print(f"Raw rows loaded: {raw_count:,}")

Step 1: Extracting raw data...
Raw rows loaded: 1,072,255


Transform

In [10]:
print("Step 2: Transforming data...")

con.execute("""
    CREATE TABLE movies_clean AS
    SELECT
        title,
        CAST(release_year AS INTEGER)                  AS release_year,
        CAST(runtime AS DOUBLE)                        AS runtime,
        CAST(budget AS DOUBLE)                         AS budget,
        CAST(revenue AS DOUBLE)                        AS revenue,
        CAST("IMDB_Rating" AS DOUBLE)                  AS imdb_rating,
        CAST(vote_count AS INTEGER)                    AS vote_count,
        original_language,
        overview,
        tagline,
        genres_list,
        "Director"                                     AS director,
        "Star1"                                        AS actor,
        "Writer"                                       AS writer,
        production_companies,
        production_countries,
        release_date,
        ROUND(((revenue - budget) / budget) * 100, 2) AS roi
    FROM movies_raw
    WHERE revenue        > 1000
    AND   budget         > 1000
    AND   runtime        > 0
    AND   release_year   >= 1990
    AND   "Director"     IS NOT NULL AND TRIM("Director") != ''
    AND   "Star1"        IS NOT NULL AND TRIM("Star1")    != ''
    AND   title          IS NOT NULL AND TRIM(title)      != ''
    AND   overview       IS NOT NULL AND TRIM(overview)   != ''
    AND   tagline        IS NOT NULL AND TRIM(tagline)    != ''
    AND   "Writer"       IS NOT NULL AND TRIM("Writer")   != ''
""")

clean_count = con.execute("SELECT COUNT(*) FROM movies_clean").fetchone()[0]
print(f"Clean rows: {clean_count:,}")
print(f"Rows removed: {raw_count - clean_count:,} ")

Step 2: Transforming data...
Clean rows: 3,146
Rows removed: 1,069,109 ✅


Verify No Nulls

In [11]:
print("Step 3: Verifying data quality...")

nulls = con.execute("""
    SELECT
        COUNT(*) - COUNT(title)       AS title_nulls,
        COUNT(*) - COUNT(director)    AS director_nulls,
        COUNT(*) - COUNT(actor)       AS actor_nulls,
        COUNT(*) - COUNT(overview)    AS overview_nulls,
        COUNT(*) - COUNT(tagline)     AS tagline_nulls,
        COUNT(*) - COUNT(revenue)     AS revenue_nulls,
        COUNT(*) - COUNT(roi)         AS roi_nulls
    FROM movies_clean
""").df()

print(nulls)
print("Null check complete ")

Step 3: Verifying data quality...
   title_nulls  director_nulls  actor_nulls  overview_nulls  tagline_nulls  \
0            0               0            0               0              0   

   revenue_nulls  roi_nulls  
0              0          0  
Null check complete 


Export as CSV + Parquet

In [12]:
print("Step 4: Exporting data...")

# Export as CSV (for Neo4j)
con.execute(f"""
    COPY movies_clean TO '{CLEAN_CSV}' (HEADER, DELIMITER ',')
""")
print(f"Saved as CSV: {CLEAN_CSV} ")

# Export as Parquet (for downstream ML/analytics)
con.execute(f"""
    COPY movies_clean TO '{CLEAN_PARQUET}' (FORMAT PARQUET)
""")
print(f"Saved as Parquet: {CLEAN_PARQUET} ")

Step 4: Exporting data...
Saved as CSV: ../data/movies_cleaned.csv 
Saved as Parquet: ../data/movies_cleaned.parquet 


Verify Exports

In [14]:
print("Step 5: Verifying exports...")

# Check CSV
csv_count = con.execute(f"""
    SELECT COUNT(*) FROM read_csv_auto('{CLEAN_CSV}')
""").fetchone()[0]
print(f"CSV rows: {csv_count:,} ")

# Check Parquet
parquet_count = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{CLEAN_PARQUET}')
""").fetchone()[0]
print(f"Parquet rows: {parquet_count:,} ")

print("\nETL Pipeline Complete! ")
print(f"Raw: {raw_count:,} → Clean: {clean_count:,} rows")
print(f"Outputs: movies_cleaned.csv + movies_cleaned.parquet")

Step 5: Verifying exports...
CSV rows: 3,146 
Parquet rows: 3,146 

ETL Pipeline Complete! 
Raw: 1,072,255 → Clean: 3,146 rows
Outputs: movies_cleaned.csv + movies_cleaned.parquet


Label Summary (Node + Relationship types)

In [15]:
print("Step 6: Extracted labels and relationship types for Neo4j...")

# Node labels
print("\nNode Labels:")
print("  - Movie")
print("  - Director")
print("  - Actor")
print("  - Genre")

# Relationship types
print("\nRelationship Types:")
print("  - DIRECTED (Director → Movie)")
print("  - ACTED_IN (Actor → Movie)")
print("  - BELONGS_TO (Movie → Genre)")
print("  - COLLABORATED_WITH (Director → Actor)")

# Column summary
cols = con.execute("DESCRIBE movies_clean").df()
print(f"\nTotal columns: {len(cols)}")
print(cols[['column_name', 'column_type']].to_string(index=False))

Step 6: Extracted labels and relationship types for Neo4j...

Node Labels:
  - Movie
  - Director
  - Actor
  - Genre

Relationship Types:
  - DIRECTED (Director → Movie)
  - ACTED_IN (Actor → Movie)
  - BELONGS_TO (Movie → Genre)
  - COLLABORATED_WITH (Director → Actor)

Total columns: 18
         column_name column_type
               title     VARCHAR
        release_year     INTEGER
             runtime      DOUBLE
              budget      DOUBLE
             revenue      DOUBLE
         imdb_rating      DOUBLE
          vote_count     INTEGER
   original_language     VARCHAR
            overview     VARCHAR
             tagline     VARCHAR
         genres_list     VARCHAR
            director     VARCHAR
               actor     VARCHAR
              writer     VARCHAR
production_companies     VARCHAR
production_countries     VARCHAR
        release_date        DATE
                 roi      DOUBLE
